In [57]:
import pandas as pd
import numpy as np


In [93]:
df = pd.read_csv("master_data_2020.csv")
df = df[~df["presvote20post"].isna()]

# ==========================================
# 1. DEFINE SOURCE COLUMNS
# ==========================================
treatment = "untrustworthy_flag"
raw_vote_choice_col = "presvote20post"

# ==========================================
# 2. OVERWRITE & CLEAN TARGET OUTCOMES FROM TEXT
# ==========================================
# Ensure we capture text responses and handle any trailing whitespaces cleanly
vote_series = df[raw_vote_choice_col].astype(str).str.strip()

# 1. Fix Biden Vote (1 if explicitly voted for Biden, 0 otherwise)
df["voted_biden_2020"] = np.where(vote_series == "Joe Biden", 1, 0)

# 2. Fix Trump Vote (1 if explicitly voted for Trump, 0 otherwise)
df["voted_trump_2020"] = np.where(vote_series == "Donald Trump", 1, 0)

# 3. Create Turnout (0 if they didn't vote or skipped, 1 if they cast a ballot)
# Based on your data, non-voters show up as "Did not vote for President" or missing strings
df["turnout_2020_binary"] = np.where(
    vote_series.str.contains("did not vote|skipped|__na__", case=False, na=True),
    0,
    1
)

# Structure our clean target dictionary for the loop
outcomes = {
    "2020 Turnout": "turnout_2020_binary",
    "Voted for Biden": "voted_biden_2020",
    "Voted for Trump": "voted_trump_2020",
}

# ==========================================
# 3. COMPREHENSIVE FEATURE ENGINEERING (CONDENSED & STREAMLINED)
# ==========================================
# Process Digital Literacy Index
tf_cols = ["tf_adv", "tf_pdf", "tf_spy", "tf_wiki", "tf_cache", "tf_phishing"]
df_tf = df[tf_cols].map(
    lambda val: int(str(val).strip().split()[0])
    if pd.notna(val) and str(val).strip().split()[0].isdigit()
    else np.nan
)
df["digital_literacy_index"] = df_tf.mean(axis=1)

# Linearized Political Mappings
df["ideo5_linear"] = df["ideo5"].map({"Very liberal": 1, "Liberal": 2, "Moderate": 3, "Conservative": 4, "Very conservative": 5})
df["pid7_linear"] = df["pid7"].map({"Strong Democrat": 1, "Not very strong Democrat": 2, "Lean Democrat": 3, "Independent": 4, "Lean Republican": 5, "Not very strong Republican": 6, "Strong Republican": 7})

# Linearized Social Media Scale
socmed_mapping = {
    "Less than 10 minutes per day": 1, "10–30 minutes per day": 2, "31–60 minutes per day": 3,
    "1–2 hours per day": 4, "2–3 hours per day": 5, "More than 3 hours per day": 6
}
df["socmed_use_linear"] = df["socmed_use"].astype(str).str.strip().map(socmed_mapping)

# --- NEW: Condense News Interest into a Linear Scale ---
newsint_mapping = {
    "Hardly at all": 1,
    "Only now and then": 2,
    "Some of the time": 3,
    "Most of the time": 4
}
df["newsint_linear"] = df["newsint"].astype(str).str.strip().map(newsint_mapping)

# --- NEW: Condense Internet Use Frequency into a Linear Scale ---
intuse_mapping = {
    "Less often": 1,
    "About once a day": 3,
    "Several times a day": 4,
    "A few times a week": 2  # Catching common top-tier survey labels if present
}
# Fallback to handle "Several times a day" as the top option if "Almost constantly" isn't used
df["intuse_linear"] = df["intuse"].astype(str).str.strip().map(intuse_mapping).fillna(3)

# --- NEW: Condense 2016 Vote into Trump, Clinton, and Other ---
# Modified 2016 vote processing function
def condense_2016_vote_fixed(val):
    val_str = str(val).strip()
    if val_str in ["Hillary Clinton", "Donald Trump"]:
        return val_str
    # Explicitly group non-voters and missing responses into a valid category string
    elif pd.isna(val) or val_str in ["__NA__", "nan", "Did not vote", "Did not vote for President"]:
        return "Did Not Vote"
    else:
        return "Other"

df["presvote16post_condensed"] = df["presvote16post"].apply(condense_2016_vote_fixed)

# REMOVE 'turnout16' from your categorical_covariates list!
# 'presvote16post_condensed' now handles turnout implicitly.
categorical_covariates = [
    "race4", 
    "educ4", 
    "region", 
    "presvote16post_condensed" # Will generate dummies for Clinton, Other, and Did Not Vote. 
]
# Gather all our dense numeric/linear features
linear_features = [
    "ideo5_linear", 
    "pid7_linear", 
    "socmed_use_linear", 
    "newsint_linear", 
    "intuse_linear"
]

# ==========================================
# 4. DESIGN MATRIX GENERATION (THE COVARIATES)
# ==========================================
core_cols = [treatment] + list(outcomes.values()) + ["age", "female"]
engineered_continuous = ["digital_literacy_index"]


# Clean complete-case alignment
all_processed_cols = core_cols + engineered_continuous + linear_features + categorical_covariates
df_clean = df[all_processed_cols].dropna().copy()

# Build the final compact X Matrix
# 1. Generate all dummies with drop_first=False so we can manually control the baselines
X_matrix = pd.get_dummies(
    df_clean[["age", "female"] + engineered_continuous + linear_features + categorical_covariates],
    columns=categorical_covariates,
    drop_first=False,
    dtype=int,
)

# 2. Define exactly one baseline reference column to drop from each categorical group
baselines_to_drop = [
    'presvote16post_condensed_Donald Trump',  # 2016 Vote baseline
    'race4_White',                            # Race baseline
    'educ4_College grad',                     # Education baseline
    'region_Midwest',                         # Region baseline
]

# 3. Drop them safely from the matrix
X_matrix = X_matrix.drop(columns=baselines_to_drop, errors='ignore')

# ==========================================
# 5. FINAL AIPW INPUT EXTRACTION
# ==========================================
W = df_clean[treatment].astype(int).values
Y_turnout = df_clean[outcomes["2020 Turnout"]].astype(int).values
Y_biden = df_clean[outcomes["Voted for Biden"]].astype(int).values
Y_trump = df_clean[outcomes["Voted for Trump"]].astype(int).values

print("--- PIPELINE VERIFICATION ---")
print(f"Total valid sample size (N): {len(df_clean)}")
print(f"Total covariates in X_matrix: {X_matrix.shape[1]}")
print(f"Mean 2020 Turnout rate: {Y_turnout.mean():.2%}")
print(f"Mean unconditional Biden support: {Y_biden.mean():.2%}")
print(f"Mean unconditional Trump support: {Y_trump.mean():.2%}")

--- PIPELINE VERIFICATION ---
Total valid sample size (N): 956
Total covariates in X_matrix: 20
Mean 2020 Turnout rate: 97.38%
Mean unconditional Biden support: 59.73%
Mean unconditional Trump support: 35.46%


In [106]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm

# 1. statsmodels requires an explicit intercept column
X_with_constant = sm.add_constant(X_matrix)

# Pack outcomes into a dictionary matching our extracted variables
outcome_data = {
    "Voted for Biden": Y_biden,
    "Voted for Trump": Y_trump
}

print("--- RUNNING LOGISTIC AIPW ESTIMATION (STATSMODELS) ---")

for name, Y in outcome_data.items():
    try:
        # ----------------------------------------------------
        # STEP 1: Propensity Score Model e(X) via pure MLE
        # ----------------------------------------------------
        prop_model = sm.Logit(W, X_with_constant).fit(disp=0, maxiter=1000)
        e_hat = prop_model.predict(X_with_constant)
        
        # Clip propensity scores to protect against extreme weights
        e_hat = np.clip(e_hat, 0.05, 0.95)
        
        # ----------------------------------------------------
        # STEP 2: Outcome Models mu_1(X) and mu_0(X)
        # ----------------------------------------------------
        # Fit model only on the treated units (W == 1)
        out_model_1 = sm.Logit(Y[W == 1], X_with_constant.iloc[W == 1]).fit(disp=0, maxiter=1000)
        mu_1_hat = out_model_1.predict(X_with_constant)
        
        # Fit model only on the control units (W == 0)
        out_model_0 = sm.Logit(Y[W == 0], X_with_constant.iloc[W == 0]).fit(disp=0, maxiter=1000)
        mu_0_hat = out_model_0.predict(X_with_constant)
        
        # ----------------------------------------------------
        # STEP 3: Compute AIPW Scores & Asymptotic Variance
        # ----------------------------------------------------
        N = len(Y)
        
        base_diff = mu_1_hat - mu_0_hat
        treated_correction = (W * (Y - mu_1_hat)) / e_hat
        control_correction = ((1 - W) * (Y - mu_0_hat)) / (1 - e_hat)
        
        # Generate final point-level scores
        aipw_scores = base_diff + treated_correction - control_correction
        
        # Unbiased Average Treatment Effect (ATE)
        ate = np.mean(aipw_scores)
        
        # Analytical standard errors derived from the empirical influence curve
        influence_curve = aipw_scores - ate
        variance = np.var(influence_curve, ddof=1) / N
        std_error = np.sqrt(variance)
        
        # Compute Wald Z-Test statistics
        z_stat = ate / std_error
        p_val = 2 * (1 - stats.norm.cdf(np.abs(z_stat)))
        
        # Confidence Intervals (95%)
        ci_lower = ate - (1.96 * std_error)
        ci_upper = ate + (1.96 * std_error)
        
        # ----------------------------------------------------
        # STEP 4: Print Clean Estimates
        # ----------------------------------------------------
        print(f"\nOutcome: {name}")
        print(f"  ATE: {ate:+.4f} ({ate*100:+.2f} percentage points)")
        print(f"  Std Error: {std_error:.4f}")
        print(f"  95% CI:   [{ci_lower:+.4f}, {ci_upper:+.4f}]")
        print(f"  p-value:   {p_val:.4f}")
        
    except Exception as e:
        print(f"\nModel for '{name}' failed to converge or encountered an error:")
        print(f"  Error details: {e}")

--- RUNNING LOGISTIC AIPW ESTIMATION (STATSMODELS) ---

Outcome: Voted for Biden
  ATE: -0.0286 (-2.86 percentage points)
  Std Error: 0.0149
  95% CI:   [-0.0578, +0.0006]
  p-value:   0.0547

Outcome: Voted for Trump
  ATE: +0.0144 (+1.44 percentage points)
  Std Error: 0.0147
  95% CI:   [-0.0144, +0.0432]
  p-value:   0.3279


In [113]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.ensemble import RandomForestClassifier

# Use our clean master X_matrix values
X = X_matrix.values
N = len(W)

candidate_outcomes = {"Voted for Biden": Y_biden, "Voted for Trump": Y_trump}

# Native NumPy K-Fold generation
np.random.seed(42)
indices = np.arange(N)
np.random.shuffle(indices)

num_folds = 5
folds = np.array_split(indices, num_folds)

print("--- RUNNING LIGHTWEIGHT RANDOM FOREST AIPW ---")

for name, Y in candidate_outcomes.items():
    # Placeholders for out-of-fold probability predictions
    e_hat = np.zeros(N)
    mu_1_hat = np.zeros(N)
    mu_0_hat = np.zeros(N)

    for f in range(num_folds):
        val_idx = folds[f]
        train_idx = np.setdiff1d(indices, val_idx)

        X_train, X_val = X[train_idx], X[val_idx]
        W_train, W_val = W[train_idx], W[val_idx]
        Y_train, Y_val = Y[train_idx], Y[val_idx]

        # ----------------------------------------------------
        # STEP 1: Lightweight Propensity Score Model e(X)
        # ----------------------------------------------------
        # Max depth = 2 prevents hyper-partisans from getting 0.99 or 0.01 propensities
        prop_model = RandomForestClassifier(
            n_estimators=100, 
            max_depth=10, 
            min_samples_leaf=10,
            random_state=42, 
            n_jobs=-1
        )
        prop_model.fit(X_train, W_train)
        e_hat[val_idx] = prop_model.predict_proba(X_val)[:, 1]

        # ----------------------------------------------------
        # STEP 2: Lightweight Outcome Models mu_1(X) and mu_0(X)
        # ----------------------------------------------------
        treated_mask = W_train == 1
        out_model_1 = RandomForestClassifier(
            n_estimators=100, 
            max_depth=10, 
            min_samples_leaf=10,
            random_state=42, 
            n_jobs=-1
        )
        out_model_1.fit(X_train[treated_mask], Y_train[treated_mask])
        mu_1_hat[val_idx] = out_model_1.predict_proba(X_val)[:, 1]

        control_mask = W_train == 0
        out_model_0 = RandomForestClassifier(
            n_estimators=100, 
            max_depth=10, 
            min_samples_leaf=10,
            random_state=42, 
            n_jobs=-1
        )
        out_model_0.fit(X_train[control_mask], Y_train[control_mask])
        mu_0_hat[val_idx] = out_model_0.predict_proba(X_val)[:, 1]

    # ----------------------------------------------------
    # STEP 3: Symmetric Trimming & AIPW Core Math
    # ----------------------------------------------------
    e_hat = np.clip(e_hat, 0.05, 0.95)

    base_diff = mu_1_hat - mu_0_hat
    treated_correction = (W * (Y - mu_1_hat)) / e_hat
    control_correction = ((1 - W) * (Y - mu_0_hat)) / (1 - e_hat)

    aipw_scores = base_diff + treated_correction - control_correction
    ate = np.mean(aipw_scores)

    # Asymptotic Standard Errors via Influence Curve
    influence_curve = aipw_scores - ate
    variance = np.var(influence_curve, ddof=1) / N
    std_error = np.sqrt(variance)

    # Inference Math
    z_stat = ate / std_error
    p_val = 2 * (1 - stats.norm.cdf(np.abs(z_stat)))
    ci_lower = ate - (1.96 * std_error)
    ci_upper = ate + (1.96 * std_error)

    print(f"\nOutcome: {name}")
    print(f"  ATE: {ate:+.4f} ({ate*100:+.2f} percentage points)")
    print(f"  Std Error: {std_error:.4f}")
    print(f"  95% CI:   [{ci_lower:+.4f}, {ci_upper:+.4f}]")
    print(f"  p-value:   {p_val:.4f}")

--- RUNNING LIGHTWEIGHT RANDOM FOREST AIPW ---

Outcome: Voted for Biden
  ATE: -0.0462 (-4.62 percentage points)
  Std Error: 0.0194
  95% CI:   [-0.0842, -0.0081]
  p-value:   0.0174

Outcome: Voted for Trump
  ATE: +0.0469 (+4.69 percentage points)
  Std Error: 0.0192
  95% CI:   [+0.0093, +0.0846]
  p-value:   0.0146
